# 03 · ONNX 流式导出 + DNS 客观质量评测

1. 导出单帧步进 ONNX，并验证 PyTorch整段 / PyTorch流式 / ONNX流式三者一致；
2. 只在 `dns_objective` 上计算 SI-SDR、STOI、ESTOI、PESQ。

AISHELL 与 WenetSpeech 的 CER 在本地 `rtse-eval` 运行；WenetSpeech 没有干净参考，
这里禁止对它计算有参考指标。


In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与开关集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 语料怎么放 ─────────────────────────────────────────────────────────
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每会话解压到本地盘。
#               一次下载永久有效；训练读取走本地盘全速。
#   'local'  —— 全在临时盘，用完即删，每个新会话都要重下。
DATA_MODE = 'hybrid'

# ── 数据规模 ───────────────────────────────────────────────────────────
# DNS5 干净语音的 split 切片数。每片 5.24 GB，实测约 **19 小时**，
# 落在"20~30 小时可管理子集"这个目标区间内。
# 切片档解压到末尾会报 EOF，属正常（详见 fetch_dns 的说明）。
N_SPEECH_SHARDS = 1
# DNS 噪声分片：audioset（日常环境声）+ freesound（标注音效）各取几片。
N_AUDIOSET_SHARDS = 2
N_FREESOUND_SHARDS = 1

# ── 快速验证模式 ───────────────────────────────────────────────────────
# V1 已停用 QUICK_TEST：两套受控集都依赖 DNS 留出噪声/RIR。
# 小规模跑通请用 SMOKE_RUN=True；QUICK_TEST 必须保持 False。
QUICK_TEST = False

# ── 小规模跑通模式 ─────────────────────────────────────────────────────
# True = 大幅缩小**用量**与**训练时长**，验证「数据→训练→导出→评测」整条链路。
# 缩的不是下载量（DNS 分片是最小单位，该下多少还是多少），而是"用多少条"和"跑多少轮"：
#   受控集   81 格 × 1 = 81 条/套（正式 5/格 = 405）
#   真实 CER 3 个时长桶 × 10 = 30 条（正式 300）
#   噪声分类 抽 400 条做平稳性判决（正式 4000）
#   训练     2000 样本/epoch × 3 epoch，只跑 crn-nano（正式 20000 × 60，两档）
# 跑通之后改成 False 再跑正式版。
SMOKE_RUN = True

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都靠它

assert DATA_MODE in ('hybrid', 'local'), 'DATA_MODE 只能是 hybrid / local'
assert not QUICK_TEST, 'V1 不支持 QUICK_TEST；请改用 SMOKE_RUN=True 做小规模闭环'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'

CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSETS_DIR = f'{DRIVE}/testsets'  # V1 三套职责分离的固定测试集
DNS_QUALITY_DIR = f'{TESTSETS_DIR}/dns_objective'
AISHELL_CER_DIR = f'{TESTSETS_DIR}/aishell_controlled'
WENET_REAL_DIR = f'{TESTSETS_DIR}/wenetspeech_real'
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致（区分大小写，空格照写）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSETS_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {DRIVE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  三套固定测试集      {TESTSETS_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(仅 WenetSpeech)" if QUICK_TEST else "完整(DNS 训练/质量 + AISHELL 受控 CER + WenetSpeech 真实 CER)"}')
print(f'  规模      {"⚡ 小规模跑通（SMOKE_RUN=True，结果不作数）" if SMOKE_RUN else "正式规模"}')
print()

!df -h /content | tail -1


In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 DRIVE_ROOT 目录下。**代码改过就要重新上传**，
# 否则 Colab 跑的还是旧逻辑（这个坑踩过，见 docs/ISSUES.md）。
ZIP = f'{DRIVE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/colab_upload/ 下的文件（zip + 3 个 notebook）传到 Drive 的 {DRIVE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(DRIVE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没预装的。不用 `pip install -e .`：那会去解析 pyproject
# 里锁定的 torch CPU 索引，把 Colab 自带的 GPU 版 torch 覆盖掉，训练慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime opencc-python-reimplemented 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
# **重新导入前必须把已加载的 rtse 从 sys.modules 里清掉。**
# 上面的 unzip 换的是磁盘上的文件，而 `import rtse` 对**已经导入过**的模块是空操作：
# 同一个 Colab 会话里重跑本 cell，磁盘上是新代码、内存里跑的还是旧的。
# 这个症状极具迷惑性——报错的行号来自旧文件，跟你手里的新文件对不上号，
# 会让人以为"包没传上去"而反复重传（见 docs/ISSUES.md I-13 / I-28）。
for _m in [m for m in list(sys.modules) if m == 'rtse' or m.startswith('rtse.')]:
    del sys.modules[_m]
import rtse
# 自证：把**实际加载的文件路径和改动时间**打出来。
# "我改的代码到底有没有在跑"必须是可观测的事实，不能靠推断（I-28）。
print('已加载 rtse ←', rtse.__file__)
print('           改动时间',
      time.strftime('%m-%d %H:%M', time.localtime(os.path.getmtime(rtse.__file__))),
      '| 若这个时间不是你刚打包的时刻，说明跑的还是旧代码')
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. 导出并校验

In [ ]:
# 正式导出必须先通过 02 的 clean透传 + 去噪正增益闸门。
gate_path = Path(f'{DRIVE}/training_gates.json')
assert gate_path.exists(), '缺少 training_gates.json；先完整运行 02 的无害性与有效性闸门'
training_gates = json.loads(gate_path.read_text(encoding='utf-8'))
if not SMOKE_RUN:
    for name, gate in training_gates.items():
        assert gate['clean_passthrough_si_sdr'] >= 20.0, f'{name} 干净透传未达标，禁止正式导出'
        assert gate['noisy_delta_si_sdr'] > 0.0, f'{name} 去噪无正增益，禁止正式导出'
print('训练闸门:', training_gates)

import torch
from rtse.models import build_model
from rtse.train.export import export_streaming_onnx

MODELS = ['crn-nano'] if SMOKE_RUN else ['crn-nano', 'crn-lite']
export_info = {}

for name in MODELS:
    ck = Path(f'{CKPT_DIR}/{name}/best.pt')
    if not ck.exists():
        print(f'[skip] {name}: 没有 best.pt，先跑 02_train.ipynb'); continue

    state = torch.load(ck, map_location='cpu', weights_only=False)
    model = build_model(name)
    model.load_state_dict(state['model'])
    model.eval()

    info = export_streaming_onnx(model, f'{MODEL_DIR}/{name}.onnx', verify=True)
    v = info['verification']
    export_info[name] = info

    print(f'\n=== {name} ===')
    print(f"  训练到 epoch {state['epoch']}，最佳 val SI-SDR {-state['best_val']:.3f} dB")
    print(f"  参数 {info['params']:,}   文件 {info['size_kb']} KB")
    print(f"  ONNX流式 vs PyTorch整段 : {v['onnx_vs_pytorch_batch']:.3e} (相对 {v['relative_error']:.3e})")
    print(f"  PyTorch流式 vs 整段     : {v['pytorch_streaming_vs_batch']:.3e}")
    print(f"  状态形状稳定            : {v['state_shape_stable']}")
    print(f"  ==> {'PASS ✅' if v['passed'] else 'FAIL ❌ 不要下载这个模型，先查因果性'}")

Path(f'{MODEL_DIR}/export_info.json').write_text(
    json.dumps(export_info, ensure_ascii=False, indent=1), encoding='utf-8')


## 2. 下载 DNSMOS 模型

DNSMOS 是**无参考** MOS 预测，是实时麦克风演示里唯一能显示的质量指标
（那里没有干净参考）。模型来自微软 DNS-Challenge 仓库，只有几 MB。

In [ ]:
os.makedirs(f'{MODEL_DIR}/dnsmos', exist_ok=True)
!wget -q -O "{MODEL_DIR}/dnsmos/sig_bak_ovr.onnx" \
  https://raw.githubusercontent.com/microsoft/DNS-Challenge/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx \
  && ls -lh "{MODEL_DIR}/dnsmos/"

# 若 404，去 https://github.com/microsoft/DNS-Challenge 的 DNSMOS 目录确认最新路径。
# 拿不到也不影响主流程：本地评测会自动把 DNSMOS 列标 n/a。

## 3. 在 DNS 客观质量集上评测（含 PESQ）

这是唯一负责 SI-SDR/STOI/PESQ 的测试集。按噪声平稳性、RIR来源和匹配后的
RT60 桶汇总；中文测试集不在这里冒充干净参考。


In [ ]:
import numpy as np
from tqdm.auto import tqdm
from rtse.audio.io import read_audio
from rtse.metrics.intrusive import si_sdr, stoi, estoi, pesq, seg_snr
from rtse.runtime import Pipeline, OnnxEnhancer
from rtse.dsp import build_dsp
from rtse.vad import build_vad

idx = json.loads(Path(f'{DNS_QUALITY_DIR}/index.json').read_text(encoding='utf-8'))
records = idx['records']
print(f'测试集 {len(records)} 个样本')

METHODS = ['none', 'specsub', 'wiener', 'mmse-lsa'] + list(export_info)

def make(method):
    if method == 'none': return None
    if method in ('specsub', 'wiener', 'mmse-lsa'): return build_dsp(method)
    return OnnxEnhancer(f'{MODEL_DIR}/{method}.onnx')

rows = []
for method in METHODS:
    enh = make(method)
    pipe = Pipeline(enhancer=enh, vad=build_vad('energy'))
    for r in tqdm(records, desc=f'{method:>10}', leave=False):
        clean = read_audio(f'{DNS_QUALITY_DIR}/{r["clean"]}')
        noisy = read_audio(f'{DNS_QUALITY_DIR}/{r["noisy"]}')
        pipe.reset()
        out, _ = pipe.process_signal(noisy)
        rows.append({'id': r['id'], 'method': method, 'snr': r['snr'],
                     'noise_kind': r['noise_kind'], 'rir_kind': r['rir_kind'],
                     'rt60_bucket': r['rt60_bucket'],
                     'rt60_measured': r['rt60_measured'],
                     'si_sdr': si_sdr(clean, out), 'seg_snr': seg_snr(clean, out),
                     'stoi': stoi(clean, out), 'estoi': estoi(clean, out),
                     'pesq': pesq(clean, out)})

Path(f'{DRIVE}/colab_metrics.json').write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
print(f'已写入 {len(rows)} 条指标 → {DRIVE}/colab_metrics.json')


In [ ]:
# 快速汇总；正式报告用本地 rtse-eval 的逐条 JSON。
import collections, statistics as st
print(f"{'method':<12}{'noise':<15}{'rir':<8}{'RT60':>7}{'SI-SDR':>9}{'STOI':>8}{'PESQ':>8}")
print('-' * 68)
agg = collections.defaultdict(list)
for r in rows:
    agg[(r['method'], r['noise_kind'], r['rir_kind'], r['rt60_bucket'])].append(r)
for key in sorted(agg, key=lambda x: tuple(str(v) for v in x)):
    m, nk, rk, rt = key
    g = agg[key]
    pq = [r['pesq'] for r in g if r['pesq'] is not None]
    print(f"{m:<12}{nk:<15}{rk:<8}{rt:>7.1f}"
          f"{st.mean(r['si_sdr'] for r in g):>9.2f}"
          f"{st.mean(r['stoi'] for r in g):>8.3f}"
          f"{(st.mean(pq) if pq else float('nan')):>8.3f}")


## 4. 打包回传

最后只需下载一个 `rtse_handoff.zip`。解压到新电脑的仓库根目录后，会直接得到：

- `models/`：本轮ONNX、导出验证信息、DNSMOS；
- `checkpoints/`：本轮模型的best/last/history，可续训；
- `data/testsets/`：三套固定测试集；
- `results/`：Colab客观指标、训练闸门、数据清单；
- `HANDOFF_MANIFEST.json`：每个文件的大小与SHA-256，供本地检查传输完整性。

不会包含几十GB的 `archives/` 原始下载缓存，也不会夹带旧模型。


In [ ]:
import datetime as dt
import hashlib
import zipfile

handoff = Path(WORK) / 'rtse_handoff'
if handoff.exists():
    shutil.rmtree(handoff)
(handoff / 'results').mkdir(parents=True)
(handoff / 'checkpoints').mkdir(parents=True)
(handoff / 'models').mkdir(parents=True)

required = {
    'models': Path(MODEL_DIR),
    'testsets': Path(TESTSETS_DIR),
    'colab_metrics': Path(DRIVE) / 'colab_metrics.json',
    'training_gates': Path(DRIVE) / 'training_gates.json',
    'data_manifest': Path(DRIVE) / 'manifest.json',
}
missing = [f'{name}: {path}' for name, path in required.items() if not path.exists()]
assert not missing, '回传包缺少必要产物：\n' + '\n'.join(missing)

shutil.copytree(required['testsets'], handoff / 'data' / 'testsets')
shutil.copy2(required['colab_metrics'], handoff / 'results' / 'colab_metrics.json')
shutil.copy2(required['training_gates'], handoff / 'results' / 'training_gates.json')
shutil.copy2(required['data_manifest'], handoff / 'results' / 'manifest.json')

# 只打包本轮配置中实际训练的模型，防止Drive里残留的旧权重/checkpoint混入。
export_info = Path(MODEL_DIR) / 'export_info.json'
dnsmos_dir = Path(MODEL_DIR) / 'dnsmos'
assert export_info.exists(), '缺少ONNX导出验证信息 export_info.json'
assert (dnsmos_dir / 'sig_bak_ovr.onnx').exists(), '缺少DNSMOS模型'
shutil.copy2(export_info, handoff / 'models' / 'export_info.json')
shutil.copytree(dnsmos_dir, handoff / 'models' / 'dnsmos')
for name in MODELS:
    onnx_path = Path(MODEL_DIR) / f'{name}.onnx'
    assert onnx_path.exists(), f'{name} ONNX不存在'
    shutil.copy2(onnx_path, handoff / 'models' / onnx_path.name)
    src = Path(CKPT_DIR) / name
    assert (src / 'best.pt').exists() and (src / 'last.pt').exists(), f'{name} checkpoint不完整'
    shutil.copytree(src, handoff / 'checkpoints' / name)

files = sorted(p for p in handoff.rglob('*') if p.is_file())
handoff_meta = {
    'schema_version': 1,
    'created_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'smoke_run': SMOKE_RUN,
    'models': list(MODELS),
    'file_count_without_manifest': len(files),
    'files': [
        {
            'path': p.relative_to(handoff).as_posix(),
            'bytes': p.stat().st_size,
            'sha256': hashlib.sha256(p.read_bytes()).hexdigest(),
        }
        for p in files
    ],
}
(handoff / 'HANDOFF_MANIFEST.json').write_text(
    json.dumps(handoff_meta, ensure_ascii=False, indent=1), encoding='utf-8')

archive = Path(DRIVE) / 'rtse_handoff.zip'
archive.unlink(missing_ok=True)
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in sorted(p for p in handoff.rglob('*') if p.is_file()):
        zf.write(path, Path('rtse_handoff') / path.relative_to(handoff))

print(f'✓ 完整回传包: {archive}')
print(f'  文件 {len(files) + 1} 个，压缩后 {archive.stat().st_size / 2**20:.1f} MB')
print('下载这一个ZIP即可；解压后把 rtse_handoff/ 内各目录合并到仓库根目录。')
